---
title: "双指针"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

## 📝 题目 11：盛最多水的容器

::: {.callout-caution}
### 📖 题目描述
给定一个长度为 `n` 的整数数组 `height`。有 `n` 条垂线，第 `i` 条线的两个端点是 `(i, 0)` 和 `(i, height[i])`。

找出其中的两条线，使得它们与 x 轴共同构成的容器可以容纳最多的水。
返回容器可以储存的最大水量。

**说明**：你不能倾斜容器。

**输入输出示例**：

* **示例 1**：
    * **输入**：`[1,8,6,2,5,4,8,3,7]`
    * **输出**：`49`
    * **解释**：图中垂直线代表输入数组 [1,8,6,2,5,4,8,3,7]。在此情况下，容器能够容纳水（表示为蓝色部分）的最大值为 49。

* **示例 2**：
    * **输入**：`height = [1,1]`
    * **输出**：`1`
:::

In [1]:
class Solution11:
    def maxArea(self, height: list[int]) -> int:
        l, r = 0, len(height) - 1
        ans = 0

        while l < r:
            current_area = (r - l) * min(height[l], height[r])  # 计算当前窗口的容积
            ans = max(ans, current_area)  # 更新历史最大值
            if height[l] < height[r]:  # 贪心移动：哪边矮，哪边往中间挪
                l += 1
            else:
                r -= 1

        return ans

In [8]:
#| code-fold: true
def test_11(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典乱序", "height": [1,8,6,2,5,4,8,3,7], "expected": 49},
        {"desc": "2. 示例 2: 只有两个柱子", "height": [1,1], "expected": 1},
        {"desc": "3. 边界: 两边极高，中间极矮", "height": [10, 1, 1, 10], "expected": 30},
        {"desc": "4. 边界: 单调递增", "height": [1, 2, 3, 4, 5], "expected": 6}, # 2*3=6
        {"desc": "5. 边界: 单调递减", "height": [5, 4, 3, 2, 1], "expected": 6},
        {"desc": "6. 山峰形状 (先增后减)", "height": [1, 3, 5, 3, 1], "expected": 6},
        {"desc": "7. 全等高度", "height": [4, 4, 4, 4], "expected": 12},
        {"desc": "8. 负落差陷阱 (中间有极高柱子但宽度不够)", "height": [1, 2, 100, 2, 1], "expected": 4}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["height"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<8} | {str(actual):<8} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_11(Solution11().maxArea)

ID  | 结果     | 预期       | 实际       | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 49       | 49       | 1. 示例 1: 经典乱序
2   | ✅ PASS | 1        | 1        | 2. 示例 2: 只有两个柱子
3   | ✅ PASS | 30       | 30       | 3. 边界: 两边极高，中间极矮
4   | ✅ PASS | 6        | 6        | 4. 边界: 单调递增
5   | ✅ PASS | 6        | 6        | 5. 边界: 单调递减
6   | ✅ PASS | 6        | 6        | 6. 山峰形状 (先增后减)
7   | ✅ PASS | 12       | 12       | 7. 全等高度
8   | ✅ PASS | 4        | 4        | 8. 负落差陷阱 (中间有极高柱子但宽度不够)
---------------------------------------------------------------------------
测试总结: 通过 8/8


::: {.callout-important}
### 💡 思路讲解

这道题如果用暴力枚举所有对 `(i, j)`，复杂度是 $O(n^2)$。但我们利用**相向双指针**，可以实现 $O(n)$ 的降维打击。

1. **确立左右边界**：
   - 初始状态下，让左指针 `l` 指向 0，右指针 `r` 指向 `n-1`。
   - 此时，窗口宽度是最大的。

2. **贪心移动逻辑（核心点）**：
   - 容器的容积公式为：$V = (r - l) \times \min(height[l], height[r])$。
   - 当我们向内移动指针时，宽度 $(r - l)$ 必然会减小。
   - **如果我们移动较高的那根柱子**：
     - 新的高度 $\min(height[l_{new}], height[r])$ 绝对不可能超过刚才的短板高度。
     - 宽度变小，高度又不增，容积 $V$ 必然会变小。所以，移动高柱子是没有任何收益的。
   - **如果我们移动较低的那根柱子（短板）**：
     - 虽然宽度变小了，但我们有可能遇到一根更高的柱子，从而弥补宽度的损失，使容积增加。
   - **结论**：每次比较左右指针的高度，**谁矮就移动谁**。

3. **动态更新**：
   - 在移动过程中，不断计算当前容积并更新全局最大值 `ans`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。左右指针从两头向中间汇合，每个元素只被访问一次。
* **空间复杂度**：$O(1)$。只使用了常数个变量存储指针和最大容积，不需要任何额外空间。
:::

## 📝 题目 15：三数之和

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums` ，判断是否存在三元组 `[nums[i], nums[j], nums[k]]` 满足 `i != j`、`i != k` 且 `j != k` ，同时还满足 `nums[i] + nums[j] + nums[k] == 0` 。

请你返回所有和为 `0` 且 **不重复** 的三元组。

**注意**：答案中不可以包含重复的三元组。

**输入输出示例**：
* **输入**：`nums = [-1,0,1,2,-1,-4]`
* **输出**：`[[-1,-1,2],[-1,0,1]]`
* **解释**：
  nums[0] + nums[1] + nums[2] = (-1) + 0 + 1 = 0
  nums[1] + nums[2] + nums[4] = 0 + 1 + (-1) = 0
  nums[0] + nums[3] + nums[4] = (-1) + 2 + (-1) = 0
  不同的三元组是 [-1,0,1] 和 [-1,-1,2] 。
:::

In [3]:
class Solution15:
    def threeSum(self, nums: list[int]) -> list[list[int]]:
        n = len(nums)
        nums.sort()  # 1. 排序：去重的基石
        ans = []

        for i in range(n):
            if nums[i] > 0:  # 2. 优化：如果当前数已经大于0，后面不可能再凑出0了
                break
            if i > 0 and nums[i] == nums[i-1]:  # 3. 去重：如果当前的数和上一个一样，跳过
                continue
            l, r = i + 1, n - 1  # 4. 双指针扫两端
            while l < r:
                s = nums[i] + nums[l] + nums[r]
                if s < 0:
                    l += 1
                elif s > 0:
                    r -= 1
                else:
                    ans.append([nums[i], nums[l], nums[r]])  # 找到一组解

                    # 5. 核心去重：跳过所有重复的数字
                    while l < r and nums[l] == nums[l+1]:
                        l += 1
                    while l < r and nums[r] == nums[r-1]:
                        r -= 1

                    # 找到解后，指针必须同时向内移动一步
                    l += 1
                    r -= 1

        return ans

In [7]:
#| code-fold: true
def test_15(func):
    test_cases = [
        {"desc": "1. 示例 1: 标准乱序包含重复", "nums": [-1,0,1,2,-1,-4], "expected": [[-1,-1,2],[-1,0,1]]},
        {"desc": "2. 全零数组", "nums": [0,0,0,0], "expected": [[0,0,0]]},
        {"desc": "3. 无解情况", "nums": [1,2,3], "expected": []},
        {"desc": "4. 只有两个数 (无法构成三元组)", "nums": [0,0], "expected": []},
        {"desc": "5. 大量重复的正负数", "nums": [-2,0,0,2,2], "expected": [[-2,0,2]]},
        {"desc": "6. 跨度极大的数值", "nums": [-10, 5, 5, 0, 1, -1], "expected": [[-10,5,5], [-1,0,1]]},
        {"desc": "7. 边界：最小元素个数", "nums": [-1,0,1], "expected": [[-1,0,1]]}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"])
        # 处理结果顺序问题，方便对比
        actual_sorted = sorted([sorted(t) for t in actual])
        expected_sorted = sorted([sorted(t) for t in tc["expected"]])

        is_pass = actual_sorted == expected_sorted
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_15(Solution15().threeSum)

ID  | 结果     | 描述
-----------------------------------------------------------------
1   | ✅ PASS | 1. 示例 1: 标准乱序包含重复
2   | ✅ PASS | 2. 全零数组
3   | ✅ PASS | 3. 无解情况
4   | ✅ PASS | 4. 只有两个数 (无法构成三元组)
5   | ✅ PASS | 5. 大量重复的正负数
6   | ✅ PASS | 6. 跨度极大的数值
7   | ✅ PASS | 7. 边界：最小元素个数
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

这道题的灵魂在于：**排序是去重的前提。**

1. **建立秩序（排序）**：
   - 首先对数组进行升序排序。排序后，相同的数字会挨在一起，方便我们跳过。

2. **固定基准（遍历 i）**：
   - 遍历数组，固定第一个数 `nums[i]`。
   - **优化**：如果 `nums[i] > 0`，由于数组已排序，后面的数肯定也大于 0，三数之和不可能为 0，直接结束循环。
   - **去重一**：如果 `nums[i] == nums[i-1]`，说明以这个数为开头的组合已经找过了，直接跳过。

3. **双指针对决（l 和 r）**：
   - 在 `i` 右侧的剩余区间 `[i + 1, n - 1]` 中，设置左右指针 `l` 和 `r`。
   - 计算 `s = nums[i] + nums[l] + nums[r]`。
   - `s < 0`：左指针右移，寻找更大的数。
   - `s > 0`：右指针左移，寻找更小的数。
   - `s == 0`：找到一组解！
     - **去重二 & 三**：在找到解后，`l` 和 `r` 必须跳过所有与当前值相同的相邻元素，直到指向不同的数，再继续收缩。

:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N^2)$。排序需要 $O(N \log N)$，外层遍历 $N$ 次，内层双指针扫描 $N$ 次，总计 $O(N^2)$。
* **空间复杂度**：$O(1)$ 或 $O(\log N)$。如果不算存储答案的空间，仅取决于排序算法的递归栈空间。
:::

## 📝 题目 713：乘积小于 K 的子数组

::: {.callout-caution}
### 📖 题目描述
给你一个正整数数组 `nums` 和一个整数 `k` 。
请你返回子数组内所有元素的乘积严格小于 `k` 的连续子数组的数目。

**注意**：`nums` 中的所有元素均为 **正整数**。

**输入输出示例**：
* **输入**：`nums = [10, 5, 2, 6], k = 100`
* **输出**：`8`
* **解释**：8 个乘积小于 100 的子数组分别为：
  [10], [5], [2], [6], [10, 5], [5, 2], [2, 6], [5, 2, 6]。
  需要注意的是 [10, 5, 2] 乘积为 100，不满足严格小于 100。

* **输入**：`nums = [1, 2, 3], k = 0`
* **输出**：`0`
:::

In [5]:
class Solution713:
    def numSubarrayProductLessThanK(self, nums: list[int], k: int) -> int:
        # 1. 边界处理：k <= 1 时不可能有满足条件的子数组
        if k <= 1:
            return 0

        product = 1
        l = 0
        ans = 0

        # 2. 移动右指针
        for r in range(len(nums)):
            product *= nums[r]

            # 3. 当乘积超标，收缩左指针
            while product >= k:
                product /= nums[l]
                l += 1

            # 4. 关键：累加当前窗口内以 r 结尾的合法子数组个数
            ans += (r - l + 1)

        return ans

In [9]:
#| code-fold: true
def test_713(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典过载收缩", "nums": [10, 5, 2, 6], "k": 100, "expected": 8},
        {"desc": "2. 示例 2: k 为 0", "nums": [1, 2, 3], "k": 0, "expected": 0},
        {"desc": "3. 边界: k 为 1", "nums": [1, 1, 1], "k": 1, "expected": 0},
        {"desc": "4. 全员合规 (极大 k)", "nums": [1, 2, 3], "k": 100, "expected": 6},
        {"desc": "5. 包含 1 的长序列", "nums": [1, 1, 1, 1], "k": 2, "expected": 10},
        {"desc": "6. 乘积刚好等于 k (不满足严格小于)", "nums": [2, 5], "k": 10, "expected": 2}, # [2], [5]
        {"desc": "7. 单个大数拦截", "nums": [10, 100, 10], "k": 50, "expected": 2} # [10], [10]
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 70)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 70)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_713(Solution713().numSubarrayProductLessThanK)

ID  | 结果     | 预期   | 实际   | 描述
----------------------------------------------------------------------
1   | ✅ PASS | 8    | 8    | 1. 示例 1: 经典过载收缩
2   | ✅ PASS | 0    | 0    | 2. 示例 2: k 为 0
3   | ✅ PASS | 0    | 0    | 3. 边界: k 为 1
4   | ✅ PASS | 6    | 6    | 4. 全员合规 (极大 k)
5   | ✅ PASS | 10   | 10   | 5. 包含 1 的长序列
6   | ✅ PASS | 2    | 2    | 6. 乘积刚好等于 k (不满足严格小于)
7   | ✅ PASS | 2    | 2    | 7. 单个大数拦截
----------------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

由于数组全是正整数，窗口扩大乘积必增，窗口缩小乘积必减。这保证了单调性。

1. **边界防御**：
   - 如果 `k <= 1`，由于数组里最小的数也是 1，任何子数组的乘积都不可能 **严格小于** 1。直接返回 0。

2. **滑动窗口吞吐**：
   - 右指针 `r` 负责向右开疆拓土，每走一步，将 `nums[r]` 乘入 `product`。
   - 如果此时 `product >= k`，左指针 `l` 开始被迫收缩，`product /= nums[l]`，直到满足条件。

3. **核心计数逻辑 (魔法公式)**：
   - **问题**：当窗口 `[l, r]` 满足条件时，新增了多少个以 `nums[r]` 为结尾的合法子数组？
   - **答案**：正是 `r - l + 1` 个。
   - **举例**：窗口是 `[5, 2, 6]`，新增以 `6` 结尾的子数组有：`[6]`, `[2, 6]`, `[5, 2, 6]`。正好是 3 个。
   - 我们累加每一轮的这个增量，就能得到最终的总数。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N)$。每个指针最多走 $N$ 步。
* **空间复杂度**：$O(1)$。只用了几个变量。
:::